In [2]:
# import libraries
from function4ReactiveTransport import *
import matplotlib.pyplot as plt
# Comment the next two lines in your actual simulations if you don't want to make animations
import matplotlib.animation as animation  # For creating animation only
plt.rcParams['animation.embed_limit'] = 5e3  # Pre-allocation of RAM in [MB] for animation
import numpy as np

# NOTE:
# This script was originally written for Jupyter (it contains IPython/matplotlib notebook patterns).
# If you run it as a plain Python script, remove Jupyter magics and IPython display imports.
try:
    from IPython.display import HTML
except Exception:
    HTML = True

In [3]:
# import solver from ReactiveTransport_solver class
solver = ReactiveTransport_solver()

In [4]:
# set computation domain
dx = 1.0  # grid size: 1 meter
Nx, Ny = 300, 100  # number of grids along x and y directions
solver.set_domain_size_discretization(Nx, Ny, dx)

########################### TASK 1
# Generate the random field
# selecting the GREEN parameter space
# Set parameters for spatial random field
mean_log10k= -12
sigma_ks = [1.0, 1.5, 1.0, 0.5, 1.0]
correlation_lengths = [15, 20, 20, 20, 25]
random_seeds = [1,2,3,4,5]


In [5]:
# check mass conservation
from matplotlib.ticker import FuncFormatter

# Define time formatting
def format_time(seconds):
    seconds = float(seconds)

    # < 2 minutes → seconds
    if seconds < 120:
        return f"{seconds:.1f} s"

    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} min"

    hours = seconds / 3600
    if hours < 24:
        return f"{hours:.2f} h"

    days = seconds / 86400
    if days < 365:
        return f"{days:.2f} d"

    years = seconds / (365 * 86400)
    return f"{years:.2f} yr"

In [6]:
# Set parameters for steady-state Darcy flow (solve the Laplace Equation)
phi = 0.2  # porosity
dP_x = 20 * 1e5  # pressure difference (in Pa) across the left and right boundaries
mu = 0.468e-3  # dynamic viscosity in (Pa s), here use water at 20 °C

# generate grid for streamplots
x_space = np.linspace(0.0, Nx * dx, int(50.0 * Nx / Ny))  # 50 is a plotting scale factor
y_space = np.linspace(0.0, Ny * dx, 50)
x_grid, y_grid = np.meshgrid(x_space, y_space, indexing='xy')

longitudinal_dispersivity = 2.0  # in meter
transversal_dispersivity = 0.25  # in meter
molecular_diffusion_coeff = 1e-8  # in m2/s

# Release particles at the desired locations
shape = 'disc'
SpecieA_lable = 'C0'  # color label ("CN" color spec) for Specie A (Reactant)
SpecieA_number = 20000  # total particle number of Specie A
SpecieA_x = [10, 5]  # desired location centered near x=10 m, radius/extent=5 m
SpecieA_y = [40, 5]  # desired location centered near y=40 m, radius/extent=5 m
SpecieB_lable = 'C9'  # color label ("CN" color spec) for Specie B (Reactant)
SpecieB_number = 20000  # total particle number of Specie B
SpecieB_x = [10, 5]  # desired location centered near x=10 m, radius/extent=5 m
SpecieB_y = [60, 5]  # desired location centered near y=60 m, radius/extent=5 m
SpecieC_lable = 'C3'  # color label ("CN" color spec) for Specie C (Product)

# Animation helper
def update_line(frame_id, label_list, lines, time_text, t_list):
    labels = [solver.color_A, solver.color_B, solver.color_C]
    for line, lab in zip(lines, labels):
        idx = (label_list[frame_id] == lab)
        line.set_data(pos_x_list[frame_id][idx], pos_y_list[frame_id][idx])

    time_text.set_text(f"t = {format_time(t_list[frame_id])}")
    return (*lines, time_text)

In [7]:
def run_simulation_noplots(sigma_k, correlation_length, random_seed):
    logk = solver.set_permeability_logk(mean_log10k, sigma_k, correlation_length, random_seed)
    vmin = logk.min()
    vmax = logk.max()

    solver.set_permeability_porosity(10.0 ** logk, phi)  # import permeability & porosity field

    # Compute fluid flow
    # (p: pressure Pa; mesh: discrete mesh; u_face & v_face: seepage velocity at faces m/s)
    p, mesh, u_face, v_face = solver.compute_flow_field(dP_x, mu)
    u_face = u_face
    v_face = v_face

    # Build fast structured interpolators (fixed mesh) and update staggered face velocities
    solver.init_face_index_maps(mesh)
    solver.update_staggered_uv(u_face, v_face)


    # evaluate u,v on plotting grid via staggered bilinear interpolation
    x_flat = x_grid.ravel()
    y_flat = y_grid.ravel()

    u_grid = solver._bilinear_u(solver.uX, x_flat, y_flat, solver.Nx, solver.Ny, solver.dx).reshape(x_grid.shape)
    v_grid = solver._bilinear_v(solver.vY, x_flat, y_flat, solver.Nx, solver.Ny, solver.dx).reshape(x_grid.shape)

    u_norm = np.sqrt(u_grid ** 2 + v_grid ** 2)

    x_center = mesh.cellCenters.value[0]
    y_center = mesh.cellCenters.value[1]

    solver.set_initial_particle_position(
        SpecieA_lable, SpecieA_number, SpecieA_x, SpecieA_y,
        SpecieB_lable, SpecieB_number, SpecieB_x, SpecieB_y,
        SpecieC_lable, shape
    )

    # Setting up solute transport with Random Walk Particle Tracking (RWPT)
    solver.set_dispersivity(longitudinal_dispersivity, transversal_dispersivity, molecular_diffusion_coeff)

    # Build dispersion gradient fields from face velocities (Ito drift correction)
    # This replaces solver.set_dispersivitiy_interpolators() from the original implementation.
    solver.update_dispersion_gradients_from_faces()

    # Compute reactive transport of particles (runtime depends on CPU)
    solver.set_time_steps(10000)
    solver.set_save_interval(20)

    # Pass face velocities, get lists (particle count decreases due to outflow x>Lx)
    pos_x_list, pos_y_list, label_list, t_list, btc_list = solver.ReactiveRandomWalk(
        u_face=u_face,
        v_face=v_face,
        CFL=0.1,
        velocity_updater=None  # supply a callable(step,t)->(u_face,v_face) for transient flow
    )

    t = np.asarray(t_list)
    NA = np.array([np.sum(lbl == solver.color_A) for lbl in label_list], dtype=int)
    NB = np.array([np.sum(lbl == solver.color_B) for lbl in label_list], dtype=int)
    NC = np.array([np.sum(lbl == solver.color_C) for lbl in label_list], dtype=int)
    Nsum = NA + NB + 2*NC

    # --- Save all relevant variables ---
    key = (sigma_k, correlation_length, random_seed)
    mesh_data = {
    "nx": mesh.nx,
    "ny": mesh.ny,
    "dx": mesh.dx,
    "dy": mesh.dy,
    "cellCenters": np.array(mesh.cellCenters.value)
    }
    results[key] = {
        "logk": logk,
        "vmin": vmin,
        "vmax": vmax,
        "p": np.array(p),
        "mesh": mesh_data,
        "u_face": np.array(u_face),
        "v_face": np.array(v_face),
        "u_grid": u_grid,
        "v_grid": v_grid,
        "u_norm": u_norm,
        "x_center": x_center,
        "y_center": y_center,
        "pos_x_list": pos_x_list,
        "pos_y_list": pos_y_list,
        "label_list": label_list,
        "t_list": t_list,
        "btc_list": btc_list,
        "t": t,
        "NA": NA,
        "NB": NB,
        "NC": NC,
        "Nsum": Nsum

    }
    

In [8]:
# # NO NEED TO RUN UNLESS sim_results.pkl FILE DOESN'T CURRENTLY EXIST IN REPOSITORY

# # producing, running, and saving simulation for 25 permeability fields
# # Initialize a results dictionary
# results = {}
# for i in range(len(sigma_ks)):
#     for seed in random_seeds:
#         run_simulation_noplots(sigma_ks[i], correlation_lengths[i], seed)

# import pickle
# import gzip

# with gzip.open('sim_results.pkl.gz', 'wb') as f:
#     pickle.dump(results, f)


In [9]:
# accessing simulation data from sim_results.pkl.gz
import pickle
import gzip
from fipy import Grid2D

with gzip.open('data/sim_results.pkl.gz', 'rb') as f:
    results = pickle.load(f)

def mesh_reconstruct(key):
    mesh_data = key["mesh"]
    mesh = Grid2D(
        dx=mesh_data["dx"],
        dy=mesh_data["dy"],
        nx=mesh_data["nx"],
        ny=mesh_data["ny"]
    )
    return mesh


In [10]:
# # how to access values
from types import SimpleNamespace

def results_access(pair_index, seed_index):
    param_key=results[(sigma_ks[pair_index], correlation_lengths[pair_index], random_seeds[seed_index])]
    mesh = mesh_reconstruct(param_key)
    logk = param_key["logk"]
    vmin = param_key["vmin"]
    vmax = param_key["vmax"]
    p = param_key["p"]
    mesh_data = param_key["mesh"]
    u_face = param_key["u_face"]
    v_face = param_key["v_face"]
    u_grid = param_key["u_grid"]
    v_grid = param_key["v_grid"]
    u_norm = param_key["u_norm"]
    x_center = param_key["x_center"]
    y_center = param_key["y_center"]
    pos_x_list = param_key["pos_x_list"]
    pos_y_list = param_key["pos_y_list"]
    label_list = param_key["label_list"]
    t_list = param_key["t_list"]
    btc_list = param_key["btc_list"]
    t = param_key["t"]
    NA = param_key["NA"]
    NB = param_key["NB"]
    NC = param_key["NC"]
    Nsum = param_key["Nsum"]

    return SimpleNamespace(
        mesh=mesh,
        logk=logk,
        vmin=vmin,
        vmax=vmax,
        p=p,
        mesh_data=mesh_data,
        u_face=u_face,
        v_face=v_face,
        u_grid=u_grid,
        v_grid=v_grid,
        u_norm=u_norm,
        x_center=x_center,
        y_center=y_center,
        pos_x_list=pos_x_list,
        pos_y_list=pos_y_list,
        label_list=label_list,
        t_list=t_list,
        btc_list=btc_list,
        t=t,
        NA=NA,
        NB=NB,
        NC=NC,
        Nsum=Nsum
    )

# results_vals = results_access(1)
# print(results_vals.logk)


In [11]:
def run_simulation_plots(pair_index, seed_index):
    results_vals = results_access(pair_index, seed_index)

    plt.figure(figsize=(15, 15 * Ny / Nx))
    plt.pcolormesh(results_vals.logk.T, vmin=results_vals.vmin, vmax=results_vals.vmax, cmap='viridis')
    plt.colorbar(orientation='vertical', shrink=0.7).ax.set_title('log(k) ')
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')
    plt.gca().set_aspect(1)

    solver.set_permeability_porosity(10.0 ** results_vals.logk, phi)  # import permeability & porosity field

    # Build fast structured interpolators (fixed mesh) and update staggered face velocities
    solver.init_face_index_maps(results_vals.mesh)
    solver.update_staggered_uv(results_vals.u_face, results_vals.v_face)

    # plotting velocity and permeability
    plt.figure(figsize=(15, 15 * Ny / Nx))
    h = plt.streamplot(x_grid, y_grid, results_vals.u_grid, results_vals.v_grid, density=[6.0, 1.5], color=np.log10(results_vals.u_norm + 1e-30), cmap='Greys_r')
    clb = plt.colorbar(h.lines, orientation='vertical', shrink=0.7)
    clb.ax.set_title('log(|u|)')
    #m2 = plt.tricontourf(x_center, y_center, logk.T.flatten(), levels=30, vmin=vmin, vmax=vmax, cmap='viridis')
    #plt.colorbar(m2,orientation='vertical', shrink=0.7).ax.set_title('log(k)')
    plt.pcolormesh(results_vals.logk.T, vmin=results_vals.vmin, vmax=results_vals.vmax, cmap='viridis')
    plt.colorbar(orientation='vertical', shrink=0.7).ax.set_title('log(k)')
    plt.gca().set_aspect(1.0)
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')

    # plotting pressure
    plt.figure(figsize=(10.8, 10.8 * Ny / Nx))
    plt.tricontourf(results_vals.x_center, results_vals.y_center, results_vals.p, cmap='YlOrBr')
    clb = plt.colorbar(orientation='vertical', shrink=0.7)
    clb.ax.set_title('Pressure (Pa)')
    plt.gca().set_aspect(1.0)
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')

    solver.set_initial_particle_position(
        SpecieA_lable, SpecieA_number, SpecieA_x, SpecieA_y,
        SpecieB_lable, SpecieB_number, SpecieB_x, SpecieB_y,
        SpecieC_lable, shape
    )

    # plot initial conditions
    plt.figure(figsize=(15, 15 * Ny / Nx))
    h = plt.streamplot(x_grid, y_grid, results_vals.u_grid, results_vals.v_grid, density=[6.0, 1.5],
                    color=np.log10(results_vals.u_norm + 1e-30), cmap='Greys_r', zorder=-1)
    clb = plt.colorbar(h.lines, orientation='vertical', shrink=0.7)
    clb.ax.set_title('log(|u|)')
    plt.scatter(solver.pos_x, solver.pos_y, c=solver.label, s=8)
    plt.gca().set_aspect(1)
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')

    # Setting up solute transport with Random Walk Particle Tracking (RWPT)
    solver.set_dispersivity(longitudinal_dispersivity, transversal_dispersivity, molecular_diffusion_coeff)

    # Build dispersion gradient fields from face velocities (Ito drift correction)
    # This replaces solver.set_dispersivitiy_interpolators() from the original implementation.
    solver.update_dispersion_gradients_from_faces()

    # Compute reactive transport of particles (runtime depends on CPU)
    solver.set_time_steps(10000)
    solver.set_save_interval(20)

    plt.figure(figsize=(9, 5))
    plt.plot(results_vals.t, results_vals.NA, label="A", lw=2)
    plt.plot(results_vals.t, results_vals.NB, label="B", lw=2)
    plt.plot(results_vals.t, results_vals.NC, label="C", lw=2)
    plt.plot(results_vals.t, results_vals.Nsum, label="A+B+2C", lw=2, ls="--", color="k")

    plt.xlabel("Time (s)")
    plt.ylabel("Number of particles")
    plt.title("Particle counts vs time")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')
    plt.tight_layout()
    plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: format_time(x)))

    # Animation helper

    # Plot last saved state
    t_id = -1

    fig, ax = plt.subplots(1, 1, figsize=(10, 10 * solver.Ly / solver.Lx))

    ax.streamplot(
        x_grid, y_grid, results_vals.u_grid, results_vals.v_grid, density=[6.0, 1.5],
        color=np.log(results_vals.u_norm + 1e-30), cmap='Greys_r', zorder=-1, linewidth=1
    )

    # Create time_text AFTER ax exists (and on the correct axes)
    time_text = ax.text(
        0.02, 0.98, "", transform=ax.transAxes,
        ha="left", va="top", fontsize=12,
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
        zorder=10
    )

    # Plot particles
    idx_A = (results_vals.label_list[t_id] == solver.color_A)
    idx_B = (results_vals.label_list[t_id] == solver.color_B)
    idx_C = (results_vals.label_list[t_id] == solver.color_C)

    line_A = ax.plot(results_vals.pos_x_list[t_id][idx_A], results_vals.pos_y_list[t_id][idx_A],
                    c=solver.color_A, marker='o', markersize=2, ls='None', alpha=0.3)
    line_B = ax.plot(results_vals.pos_x_list[t_id][idx_B], results_vals.pos_y_list[t_id][idx_B],
                    c=solver.color_B, marker='o', markersize=2, ls='None', alpha=0.3)
    line_C = ax.plot(results_vals.pos_x_list[t_id][idx_C], results_vals.pos_y_list[t_id][idx_C],
                    c=solver.color_C, marker='o', markersize=2, ls='None', alpha=0.1)

    ax.set_xlim(0.0, solver.Lx)
    ax.set_ylim(0.0, solver.Ly)
    ax.set_aspect('equal')
    plt.title(f'Plot for (sigma={sigma_ks[pair_index]}, L={correlation_lengths[pair_index]}), seed={random_seeds[seed_index]}')

    # Set initial time so you see it even before the first animation update
    time_text.set_text(f"t = {format_time(results_vals.t_list[t_id])}")


    ani = animation.FuncAnimation(
        fig, update_line, frames=len(results_vals.pos_x_list),
        fargs=(results_vals.label_list, [line_A[0], line_B[0], line_C[0]], time_text, results_vals.t_list),
        interval=70, blit=False
    )
    
    plt.show()

    

In [12]:
##### Task 1, run simulations for all 25 permeability fields (takes 2.5 min to run)
# for i in range(len(sigma_ks)):
#     for j in range(len(random_seeds)):
#         run_simulation_plots(i,j)


In [ ]:
######### Task 2, Steady-state fluid velocity computation for each permeability field

# darcy velocities already calculated in earlier simulation
# -->given by u_face and v_face (flux components at faces)
# and u_grid and v_grid (darcy velocity field)
# import numpy as np

# fluid_vels = {}

# for i in range(len(sigma_ks)):
#     for j in range(len(random_seeds)):
#         results_vals = results_access(i,j)
#         param_key = (sigma_ks[i], correlation_lengths[i], random_seeds[j])

#         # pore velocity field
#         u_mean_grid = results_vals.u_grid / phi
#         v_mean_grid = results_vals.v_grid / phi
#         # magnitude of mean velocity
#         u_mean_norm = np.sqrt(u_mean_grid**2 + v_mean_grid**2)

#         # pore velocity field at faces
#         u_face_mean = results_vals.u_face / phi
#         v_face_mean = results_vals.v_face / phi

#         #saving values
#         fluid_vels[param_key] = {
#             'u_face_mean': u_face_mean,
#             'v_face_mean': v_face_mean,
#             'u_mean_grid': u_mean_grid,
#             'v_mean_grid': v_mean_grid,
#             'u_mean_norm': u_mean_norm
#         }

# import pickle

# with open('data/fluid_velocities.pkl', 'wb') as f:
#     pickle.dump(fluid_vels, f)


In [ ]:
# accessing fluid velocities
import pickle

with open('data/fluid_velocities.pkl', 'rb') as f:
    fluid_vels = pickle.load(f)

# To access u_face_mean for a specific parameter set:
i=1
j=1
param_key = (sigma_ks[i], correlation_lengths[i], random_seeds[j])
#u_face_mean = fluid_vels[param_key]['u_face_mean']

print(u_face_mean)

KeyError: 'u_face_mean'

In [26]:
fluid_vels[param_key]

{'logk': array([[-11.82231034, -11.77285358, -11.72250265, ..., -12.08672731,
         -12.06263358, -12.04360961],
        [-11.73794195, -11.69129524, -11.64409931, ..., -12.0211878 ,
         -11.99917   , -11.98270945],
        [-11.65558719, -11.61198672, -11.56818381, ..., -11.95242405,
         -11.93232505, -11.91825048],
        ...,
        [-12.25980616, -12.21444657, -12.16950722, ..., -12.5865645 ,
         -12.61328915, -12.6350573 ],
        [-12.25710403, -12.21046341, -12.16407844, ..., -12.54890435,
         -12.57542379, -12.59717446],
        [-12.25292706, -12.20546274, -12.15807937, ..., -12.50776833,
         -12.53406089, -12.5557873 ]], shape=(300, 100)),
 'vmin': np.float64(-14.265463481043424),
 'vmax': np.float64(-10.286418905258786),
 'p': array([1.99798288e+06, 1.99428265e+06, 1.99117272e+06, ...,
        3.46599808e+03, 1.98734920e+03, 6.40740490e+02], shape=(30000,)),
 'mesh': {'nx': 300,
  'ny': 100,
  'dx': np.float64(1.0),
  'dy': 1.0,
  'cellCenters'

In [14]:
# This command writes the plots directly to the notebook, so it increases the file size.
# Uncomment the following line to generate the animation in Jupyter.
# if HTML is not None:
#HTML(ani.to_jshtml())